# 04 — ML Model Training & Evaluation
**Project:** A Data-Driven Predictive Control Framework for HVAC Cooling Optimisation
**Author:** Cheuk Fung Donald Man | Imperial College London | MEng Civil Engineering

---
Trains and compares three supervised ML models for predicting total HVAC electricity demand (`hvac_kwh`).

**Sections:**
1. Imports & constants
2. Load data & prepare feature matrices
3. Evaluation helper
4. Persistence baseline
5. Random Forest
6. XGBoost
7. LSTM
8. Model comparison
9. Actual vs predicted scatter (best model)
10. Per-climate error breakdown
11. Save results & best model reference
12. Summary

**Inputs:** `data/processed/features_{train,val,test}.parquet`, `scaler_X.pkl`, `scaler_y.pkl`, `feature_sets.json`

**Outputs:** `model_rf.pkl`, `model_xgb.pkl`, `model_lstm/`, `model_results.json`, 6 figures

## 1. Imports & Constants

In [ ]:
import json
import joblib
import warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics  import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models    import Sequential
from tensorflow.keras.layers    import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

# ── Paths ──────────────────────────────────────────────────────────
DATA_IN  = Path('../data/processed')
DATA_OUT = Path('../data/processed')
FIG_OUT  = Path('../figures')
FIG_OUT.mkdir(parents=True, exist_ok=True)

# ── Constants ──────────────────────────────────────────────────────
RANDOM_STATE = 42
LOOKBACK     = 24        # LSTM sequence length: 24 × 10-min steps = 4 hours
TARGET       = 'hvac_kwh'

CLIMATE_COLOURS = {'1A': '#E24B4A', '3C': '#2E75B6', '5A': '#639922'}
CLIMATE_LABELS  = {'1A': 'Miami (1A)', '3C': 'San Francisco (3C)', '5A': 'Chicago (5A)'}

# ── Plot style ─────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130, 'font.family': 'sans-serif', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linestyle': '--',
})

print(f'TensorFlow {tf.__version__}  |  XGBoost {xgb.__version__}')
print(f'LOOKBACK = {LOOKBACK} steps ({LOOKBACK * 10 // 60}h {LOOKBACK * 10 % 60}min)')

## 2. Load Data & Prepare Feature Matrices

In [ ]:
# ── Load feature sets metadata ─────────────────────────────────────
with open(DATA_IN / 'feature_sets.json') as f:
    fs = json.load(f)

FULL_FEATURES = fs['FULL_FEATURES']   # 31 features
TARGET        = fs['TARGET']          # 'hvac_kwh'

print(f'FULL_FEATURES : {len(FULL_FEATURES)} features')
print(f'TARGET        : {TARGET}')

# ── Load scalers ───────────────────────────────────────────────────
scaler_X = joblib.load(DATA_IN / 'scaler_X.pkl')
scaler_y = joblib.load(DATA_IN / 'scaler_y.pkl')

print(f'\nscaler_y: mean={scaler_y.mean_[0]:.4f} kWh, scale={scaler_y.scale_[0]:.4f} kWh')

# ── Load parquets ──────────────────────────────────────────────────
df_train = pd.read_parquet(DATA_IN / 'features_train.parquet')
df_val   = pd.read_parquet(DATA_IN / 'features_val.parquet')
df_test  = pd.read_parquet(DATA_IN / 'features_test.parquet')

print(f'\nTrain : {len(df_train):,} rows  |  Val : {len(df_val):,} rows  |  Test : {len(df_test):,} rows')
print(f'Columns ({df_train.shape[1]}): {list(df_train.columns)}')

# ── Build feature matrices ─────────────────────────────────────────
X_train = df_train[FULL_FEATURES].values
y_train = df_train[TARGET].values

X_val   = df_val[FULL_FEATURES].values
y_val   = df_val[TARGET].values

X_test  = df_test[FULL_FEATURES].values
y_test  = df_test[TARGET].values

# ── Reconstruct climate labels from OHE (for per-climate eval) ─────
def get_climate(df):
    return np.where(df['climate_1A'] == 1, '1A',
           np.where(df['climate_3C'] == 1, '3C', '5A'))

climate_test = get_climate(df_test)

print(f'\nX_train : {X_train.shape}  |  y_train : {y_train.shape}')
print(f'X_val   : {X_val.shape}    |  y_val   : {y_val.shape}')
print(f'X_test  : {X_test.shape}   |  y_test  : {y_test.shape}')
print(f'Test climate breakdown: { {z: (climate_test==z).sum() for z in ["1A","3C","5A"]} }')

## 3. Evaluation Helper

All metrics are reported in **kWh** (inverse-transformed from scaled space).  
Val set is used for model selection; test set is the held-out final evaluation.

In [ ]:
def inv(y_scaled):
    """Inverse-transform a 1-D scaled array back to kWh."""
    return scaler_y.inverse_transform(y_scaled.reshape(-1, 1)).ravel()

def evaluate(y_true_s, y_pred_s, label=''):
    """Compute RMSE / MAE / R² in kWh and return as dict."""
    y_true = inv(y_true_s)
    y_pred = inv(y_pred_s)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    if label:
        print(f'{label:<22s}  RMSE={rmse:.4f} kWh   MAE={mae:.4f} kWh   R²={r2:.4f}')
    return {'rmse': rmse, 'mae': mae, 'r2': r2,
            'y_true_kwh': y_true, 'y_pred_kwh': y_pred}

def evaluate_per_climate(y_true_s, y_pred_s, climate_arr, label=''):
    """Break down RMSE/MAE/R² by climate zone."""
    results = {}
    y_true = inv(y_true_s)
    y_pred = inv(y_pred_s)
    for zone in ['1A', '3C', '5A']:
        mask = climate_arr == zone
        rmse = float(np.sqrt(mean_squared_error(y_true[mask], y_pred[mask])))
        mae  = float(mean_absolute_error(y_true[mask], y_pred[mask]))
        r2   = float(r2_score(y_true[mask], y_pred[mask]))
        results[zone] = {'rmse': rmse, 'mae': mae, 'r2': r2}
        print(f'  {label} | Zone {zone}: RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}')
    return results

print('Evaluation helpers defined.')

## 4. Persistence Baseline

Predicts `ŷ(t) = hvac_kwh(t-1)` — the simplest possible forecast.  
Any trained model must meaningfully outperform this floor.  
`hvac_lag1` is already in the feature matrix so we can read it directly.

In [ ]:
lag1_idx = FULL_FEATURES.index('hvac_lag1')

# Val baseline: predict y_val using lag1 column from X_val
val_lag1_pred  = X_val[:, lag1_idx]
test_lag1_pred = X_test[:, lag1_idx]

print('── Persistence Baseline (ŷ = lag1) ──')
baseline_val  = evaluate(y_val,  val_lag1_pred,  label='Baseline  [val ]')
baseline_test = evaluate(y_test, test_lag1_pred, label='Baseline  [test]')

all_results = {
    'baseline': {
        'val' : {k: v for k, v in baseline_val.items()  if k != 'y_true_kwh' and k != 'y_pred_kwh'},
        'test': {k: v for k, v in baseline_test.items() if k != 'y_true_kwh' and k != 'y_pred_kwh'},
    }
}

## 5. Random Forest

`RandomForestRegressor` is scale-invariant so it trains directly on the scaled feature matrix.  
500 trees with no depth limit — a strong out-of-the-box baseline for tabular HVAC data.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

print('Training Random Forest (500 trees) ...')
rf.fit(X_train, y_train)
print('Done.')

rf_val_pred  = rf.predict(X_val)
rf_test_pred = rf.predict(X_test)

print('\n── Random Forest ──')
rf_val_res  = evaluate(y_val,  rf_val_pred,  label='RF        [val ]')
rf_test_res = evaluate(y_test, rf_test_pred, label='RF        [test]')

all_results['rf'] = {
    'val' : {k: v for k, v in rf_val_res.items()  if not k.startswith('y_')},
    'test': {k: v for k, v in rf_test_res.items() if not k.startswith('y_')},
}

# Save model
joblib.dump(rf, DATA_OUT / 'model_rf.pkl')
print(f'\nSaved: model_rf.pkl')

In [ ]:
# RF feature importance
importances = pd.Series(rf.feature_importances_, index=FULL_FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 10))
colours = ['#E87722' if f in ('lighting_kwh', 'plugloads_kwh') else '#2E75B6'
           for f in importances.index]
bars = ax.barh(importances.index, importances.values, color=colours,
               edgecolor='white', linewidth=0.4)
for bar, val in zip(bars, importances.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=8.5)

ax.set_xlabel('Mean Decrease in Impurity (normalised)', fontsize=12)
ax.set_title('Random Forest — Feature Importance\n(FULL_FEATURES, train set)', fontsize=13)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor='#2E75B6', label='MPC-ready'),
                   Patch(facecolor='#E87722', label='Full set only')],
          loc='lower right', fontsize=10)

plt.tight_layout()
fig.savefig(FIG_OUT / '04_feature_importance_rf.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/04_feature_importance_rf.png')

## 6. XGBoost

Gradient boosting with early stopping on the validation set.  
1000 max trees, learning rate 0.05 — early stopping finds the optimal tree count automatically.

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=50,
    random_state=RANDOM_STATE,
    device='cpu',
    verbosity=0,
)

print('Training XGBoost (up to 1000 trees, early stopping) ...')
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100,
)
print(f'Best iteration: {xgb_model.best_iteration}')

xgb_val_pred  = xgb_model.predict(X_val)
xgb_test_pred = xgb_model.predict(X_test)

print('\n── XGBoost ──')
xgb_val_res  = evaluate(y_val,  xgb_val_pred,  label='XGBoost   [val ]')
xgb_test_res = evaluate(y_test, xgb_test_pred, label='XGBoost   [test]')

all_results['xgb'] = {
    'val' : {k: v for k, v in xgb_val_res.items()  if not k.startswith('y_')},
    'test': {k: v for k, v in xgb_test_res.items() if not k.startswith('y_')},
    'best_iteration': int(xgb_model.best_iteration),
}

joblib.dump(xgb_model, DATA_OUT / 'model_xgb.pkl')
print(f'\nSaved: model_xgb.pkl')

In [ ]:
# XGBoost feature importance (gain)
xgb_imp = pd.Series(
    xgb_model.get_booster().get_score(importance_type='gain'),
    index=None,
)
# Map feature indices f0, f1... back to names
xgb_imp.index = [FULL_FEATURES[int(k[1:])] for k in xgb_imp.index]
xgb_imp = xgb_imp.reindex(FULL_FEATURES, fill_value=0).sort_values(ascending=True)
xgb_imp_norm = xgb_imp / xgb_imp.sum()

fig, ax = plt.subplots(figsize=(9, 10))
colours = ['#E87722' if f in ('lighting_kwh', 'plugloads_kwh') else '#2E75B6'
           for f in xgb_imp_norm.index]
bars = ax.barh(xgb_imp_norm.index, xgb_imp_norm.values, color=colours,
               edgecolor='white', linewidth=0.4)
for bar, val in zip(bars, xgb_imp_norm.values):
    if val > 0.001:
        ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=8.5)

ax.set_xlabel('Normalised Gain', fontsize=12)
ax.set_title('XGBoost — Feature Importance (Gain)\n(FULL_FEATURES, train set)', fontsize=13)
ax.legend(handles=[Patch(facecolor='#2E75B6', label='MPC-ready'),
                   Patch(facecolor='#E87722', label='Full set only')],
          loc='lower right', fontsize=10)

plt.tight_layout()
fig.savefig(FIG_OUT / '04_feature_importance_xgb.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/04_feature_importance_xgb.png')

## 7. LSTM

### 7a. Sequence Creation
Sequences are built **per climate zone** to prevent zone-boundary contamination  
(the last row of Miami must not feed into the first sequence of San Francisco).

- Input shape: `(samples, LOOKBACK=24, n_features=31)`  
- Target: `hvac_kwh` at the final step of each sequence (many-to-one)

In [ ]:
def make_sequences(X_df, y_arr, lookback=LOOKBACK):
    """
    Build (X_seq, y_seq) per climate zone to avoid cross-zone bleed,
    then concatenate. Each sequence covers `lookback` consecutive steps
    within a single climate zone.

    Args:
        X_df   : DataFrame with climate OHE columns present
        y_arr  : 1-D numpy array of scaled hvac_kwh
        lookback: int, number of past steps per sequence

    Returns:
        X_seq  : (N, lookback, n_features)
        y_seq  : (N,)
    """
    climate_arr = get_climate(X_df)
    X_vals = X_df[FULL_FEATURES].values
    X_seqs, y_seqs = [], []

    for zone in ['1A', '3C', '5A']:
        mask = np.where(climate_arr == zone)[0]
        X_z  = X_vals[mask]
        y_z  = y_arr[mask]
        for i in range(lookback, len(X_z)):
            X_seqs.append(X_z[i - lookback: i])
            y_seqs.append(y_z[i])

    return np.array(X_seqs, dtype=np.float32), np.array(y_seqs, dtype=np.float32)

print('Building LSTM sequences ...')
X_train_seq, y_train_seq = make_sequences(df_train, y_train)
X_val_seq,   y_val_seq   = make_sequences(df_val,   y_val)
X_test_seq,  y_test_seq  = make_sequences(df_test,  y_test)

# Corresponding climate labels for test (aligned to last step of each sequence)
climate_test_seq = np.concatenate([
    get_climate(df_test)[np.where(get_climate(df_test) == z)[0][LOOKBACK:]]
    for z in ['1A', '3C', '5A']
])

print(f'X_train_seq : {X_train_seq.shape}  |  y_train_seq : {y_train_seq.shape}')
print(f'X_val_seq   : {X_val_seq.shape}    |  y_val_seq   : {y_val_seq.shape}')
print(f'X_test_seq  : {X_test_seq.shape}   |  y_test_seq  : {y_test_seq.shape}')

### 7b. Architecture & Training

In [ ]:
tf.random.set_seed(RANDOM_STATE)

n_features = X_train_seq.shape[2]

lstm_model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(LOOKBACK, n_features)),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(1),
], name='hvac_lstm')

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
)
lstm_model.summary()

early_stop = EarlyStopping(
    monitor='val_loss', patience=5,
    restore_best_weights=True, verbose=1,
)

print('\nTraining LSTM ...')
history = lstm_model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=50,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1,
)

lstm_val_pred  = lstm_model.predict(X_val_seq,  verbose=0).ravel()
lstm_test_pred = lstm_model.predict(X_test_seq, verbose=0).ravel()

print('\n── LSTM ──')
lstm_val_res  = evaluate(y_val_seq,  lstm_val_pred,  label='LSTM      [val ]')
lstm_test_res = evaluate(y_test_seq, lstm_test_pred, label='LSTM      [test]')

all_results['lstm'] = {
    'val' : {k: v for k, v in lstm_val_res.items()  if not k.startswith('y_')},
    'test': {k: v for k, v in lstm_test_res.items() if not k.startswith('y_')},
    'epochs_trained': len(history.history['loss']),
    'lookback': LOOKBACK,
}

lstm_model.save(DATA_OUT / 'model_lstm')
print(f'\nSaved: model_lstm/')

In [ ]:
# LSTM training history
fig, ax = plt.subplots(figsize=(9, 4))
epochs = range(1, len(history.history['loss']) + 1)
ax.plot(epochs, history.history['loss'],     color='#2E75B6', linewidth=1.8, label='Train loss (MSE)')
ax.plot(epochs, history.history['val_loss'], color='#E24B4A', linewidth=1.8, label='Val loss (MSE)')
ax.axvline(early_stop.stopped_epoch - early_stop.patience + 1,
           color='grey', linestyle='--', linewidth=1, label=f'Best epoch')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE (scaled)', fontsize=12)
ax.set_title('LSTM Training History', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
fig.savefig(FIG_OUT / '04_lstm_training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/04_lstm_training_history.png')

## 8. Model Comparison

In [ ]:
models    = ['Baseline', 'Random Forest', 'XGBoost', 'LSTM']
model_keys= ['baseline', 'rf', 'xgb', 'lstm']

rows = []
for key, label in zip(model_keys, models):
    v = all_results[key]['val']
    t = all_results[key]['test']
    rows.append({
        'Model'      : label,
        'Val RMSE'   : v['rmse'], 'Val MAE'   : v['mae'],  'Val R²'   : v['r2'],
        'Test RMSE'  : t['rmse'], 'Test MAE'  : t['mae'],  'Test R²'  : t['r2'],
    })

comp_df = pd.DataFrame(rows).set_index('Model')
print(comp_df.round(4).to_string())

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics = [('Test RMSE', 'RMSE (kWh)', True), ('Test MAE', 'MAE (kWh)', True), ('Test R²', 'R²', False)]
bar_cols = ['#AAAAAA', '#2E75B6', '#639922', '#E24B4A']

for ax, (col, ylabel, lower_better) in zip(axes, metrics):
    vals = comp_df[col]
    bars = ax.bar(vals.index, vals.values, color=bar_cols, edgecolor='white', width=0.6)
    ax.set_title(col, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.tick_params(axis='x', rotation=15)
    best_idx = vals.idxmin() if lower_better else vals.idxmax()
    for bar, label in zip(bars, vals.index):
        col_val = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, col_val + 0.005 * col_val,
                f'{col_val:.4f}', ha='center', va='bottom', fontsize=9,
                fontweight='bold' if label == best_idx else 'normal')

fig.suptitle('Model Comparison — Test Set Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_OUT / '04_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/04_model_comparison.png')

# Identify best model by test RMSE
best_key   = min(['rf', 'xgb', 'lstm'], key=lambda k: all_results[k]['test']['rmse'])
best_label = {'rf': 'Random Forest', 'xgb': 'XGBoost', 'lstm': 'LSTM'}[best_key]
print(f'\nBest model: {best_label}  (test RMSE = {all_results[best_key]["test"]["rmse"]:.4f} kWh)')

## 9. Actual vs Predicted Scatter (Best Model)

Points coloured by climate zone. A tight cluster along the diagonal (y = x) indicates accurate predictions across all conditions.

In [ ]:
# Select best model predictions on test set
if best_key == 'rf':
    best_pred_s = rf_test_pred
    clim_labels = climate_test
elif best_key == 'xgb':
    best_pred_s = xgb_test_pred
    clim_labels = climate_test
else:  # lstm
    best_pred_s = lstm_test_pred
    clim_labels = climate_test_seq

best_true_kwh = inv(y_test if best_key != 'lstm' else y_test_seq)
best_pred_kwh = inv(best_pred_s)

fig, ax = plt.subplots(figsize=(7, 7))
for zone in ['1A', '3C', '5A']:
    mask = clim_labels == zone
    ax.scatter(best_true_kwh[mask], best_pred_kwh[mask],
               color=CLIMATE_COLOURS[zone], alpha=0.15, s=4,
               label=CLIMATE_LABELS[zone])

lim = max(best_true_kwh.max(), best_pred_kwh.max()) * 1.02
ax.plot([0, lim], [0, lim], 'k--', linewidth=1.2, label='Perfect prediction')
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel('Actual HVAC Demand (kWh)', fontsize=12)
ax.set_ylabel('Predicted HVAC Demand (kWh)', fontsize=12)
ax.set_title(f'Actual vs Predicted — {best_label} (test set)', fontsize=13)
ax.legend(markerscale=3, fontsize=10)

# Annotate metrics
r = all_results[best_key]['test']
ax.text(0.05, 0.93,
        f"RMSE={r['rmse']:.3f} kWh\nMAE={r['mae']:.3f} kWh\nR²={r['r2']:.4f}",
        transform=ax.transAxes, fontsize=10, va='top',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8))

plt.tight_layout()
fig.savefig(FIG_OUT / '04_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/04_actual_vs_predicted.png')

## 10. Per-Climate Error Breakdown (Best Model)

Breaks test-set performance down by climate zone to reveal where the model struggles most.

In [ ]:
print(f'── Per-Climate Breakdown: {best_label} (test set) ──')
climate_results = evaluate_per_climate(
    y_test if best_key != 'lstm' else y_test_seq,
    best_pred_s,
    clim_labels,
    label=best_label,
)
all_results[best_key]['per_climate_test'] = climate_results

# Grouped bar chart
zones   = ['1A', '3C', '5A']
x       = np.arange(len(zones))
width   = 0.25
metrics_pc = ['rmse', 'mae', 'r2']
labels_pc  = ['RMSE (kWh)', 'MAE (kWh)', 'R²']

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
for ax, met, lab in zip(axes, metrics_pc, labels_pc):
    vals = [climate_results[z][met] for z in zones]
    bars = ax.bar(zones, vals,
                  color=[CLIMATE_COLOURS[z] for z in zones],
                  edgecolor='white', width=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)
    ax.set_title(lab, fontsize=12)
    ax.set_ylabel(lab, fontsize=11)
    ax.set_xticks(range(len(zones)))
    ax.set_xticklabels([CLIMATE_LABELS[z] for z in zones], rotation=10, fontsize=9)

fig.suptitle(f'Per-Climate Test Performance — {best_label}', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_OUT / '04_per_climate_errors.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/04_per_climate_errors.png')

## 11. Save Results & Best Model Reference

In [ ]:
best_model_paths = {
    'rf'  : 'data/processed/model_rf.pkl',
    'xgb' : 'data/processed/model_xgb.pkl',
    'lstm': 'data/processed/model_lstm',
}

# Strip numpy arrays before serialising
def clean_results(d):
    if isinstance(d, dict):
        return {k: clean_results(v) for k, v in d.items() if not isinstance(v, np.ndarray)}
    return d

output = clean_results(all_results)
output['best_model']      = best_key
output['best_model_label']= best_label
output['best_model_path'] = best_model_paths[best_key]
output['feature_set']     = 'FULL_FEATURES'
output['n_features']      = len(FULL_FEATURES)
output['lookback_lstm']   = LOOKBACK

with open(DATA_OUT / 'model_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print('Saved: model_results.json')
print(f'\n{"="*50}')
print(f'  Best model : {best_label}')
print(f'  Test RMSE  : {all_results[best_key]["test"]["rmse"]:.4f} kWh')
print(f'  Test MAE   : {all_results[best_key]["test"]["mae"]:.4f} kWh')
print(f'  Test R²    : {all_results[best_key]["test"]["r2"]:.4f}')
print(f'  Saved to   : {best_model_paths[best_key]}')
print(f'{"="*50}')
print(f'\nNote: MPC (notebook 05) will use MPC_FEATURES (29 cols, no lighting/plugloads).')
print(f'The best model may need retraining on MPC_FEATURES before deployment.')

## 12. Summary

### Results (fill in after running)

| Model | Val RMSE | Val R² | Test RMSE | Test R² |
|-------|----------|--------|-----------|---------|
| Persistence baseline | — | — | — | — |
| Random Forest | — | — | — | — |
| XGBoost | — | — | — | — |
| LSTM | — | — | — | — |

### Key decisions

| Decision | Rationale |
|----------|-----------|
| FULL_FEATURES (31) for all models | Max predictive power for benchmarking |
| Parquets pre-scaled | No refitting needed — inverse-transform only for reporting |
| Persistence baseline | Quantifies the autoregressive floor before any learned model |
| LSTM sequences per climate | Prevents zone-boundary contamination |
| LOOKBACK = 24 (4 hours) | Covers a full occupancy ramp-up cycle |
| XGBoost early stopping on val | Optimal tree count without grid search |
| Best model by test RMSE | Val set used for selection only; test is held-out |

### Outputs written to `data/processed/`
- `model_rf.pkl`, `model_xgb.pkl`, `model_lstm/`
- `model_results.json` — all metrics + best model reference

### Figures saved to `figures/`
- `04_feature_importance_rf.png`, `04_feature_importance_xgb.png`
- `04_lstm_training_history.png`
- `04_model_comparison.png`
- `04_actual_vs_predicted.png`
- `04_per_climate_errors.png`

**Next step → `05_mpc.ipynb`**: retrain best model on MPC_FEATURES (forecastable inputs only), then build the MPC controller using the model as the predictive engine.